# ServiceTitan Jobs Scraper

**Source:** [servicetitan.com/careers](https://www.servicetitan.com/careers)  
**Backend ATS:** Workday (`servicetitan.wd1.myworkdayjobs.com`)  
**Filter:** Armenia / Yerevan positions only

### Scraping strategy
1. Use Workday's public CXS search API to list all jobs with pagination (no auth needed)
2. Filter to Armenia/Yerevan jobs from the listing response
3. Use **Playwright** (headless Chromium) to render each job detail page — Workday is a fully client-side SPA; `requests` alone cannot retrieve job descriptions

### Why Playwright?
Workday's `/jobdetails` REST endpoint returns HTTP 500 for server-side requests (requires browser session state). The HTML page is empty without JavaScript execution. Playwright renders the page fully and waits for the job description element to load.

### Ethics & robots.txt
- Workday public job board is designed for public access
- No authentication required for public job listings
- Rate-limited: 1.5 s between page loads

### Output files
- `data/raw/jobs/servicetitan_jobs_raw.csv`
- `data/processed/jobs/servicetitan_jobs_standardized.csv`

In [1]:
import re
import time
import asyncio
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from playwright.async_api import async_playwright

BASE_WD   = "https://servicetitan.wd1.myworkdayjobs.com"
LIST_URL  = f"{BASE_WD}/wday/cxs/servicetitan/ServiceTitan/jobs"
HEADERS   = {"User-Agent": "Mozilla/5.0 (compatible; ThesisResearch/1.0; academic use)",
             "Content-Type": "application/json"}
DELAY_S   = 1.5

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"Workday board: {LIST_URL}")
print(f"Raw output:  {RAW_DIR}")
print(f"Proc output: {PROC_DIR}")

Workday board: https://servicetitan.wd1.myworkdayjobs.com/wday/cxs/servicetitan/ServiceTitan/jobs
Raw output:  /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc output: /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper

In [2]:
def html_to_text(html_str):
    if not html_str: return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

print("Helpers defined.")

Helpers defined.


## Step 1 — Discover Armenia jobs via Workday CXS API

In [3]:
all_jobs = []
offset = 0
while True:
    data = requests.post(
        LIST_URL, headers=HEADERS,
        json={"appliedFacets": {}, "limit": 20, "offset": offset, "searchText": ""},
        timeout=20
    ).json()
    batch = data.get("jobPostings", [])
    if not batch: break
    all_jobs.extend(batch)
    if len(all_jobs) >= data.get("total", 0): break
    offset += 20
    time.sleep(0.5)

armenia = [j for j in all_jobs
           if "armenia" in j.get("locationsText","").lower()
           or "yerevan" in j.get("locationsText","").lower()]

print(f"Total jobs on board: {len(all_jobs)}")
print(f"Armenia jobs: {len(armenia)}")
print()
for j in armenia:
    print(f"  {j['title']:55s} | {j['locationsText']}")

Total jobs on board: 40
Armenia jobs: 4

  Senior Facilities Specialist                            | Yerevan, Armenia
  Customer Success Manager                                | Yerevan, Armenia
  Associate Data Quality Analyst                          | Yerevan, Armenia
  Staff Site Reliability Engineer(Database)               | Yerevan, Armenia


## Step 2 — Scrape job descriptions with Playwright

Workday job pages are client-side rendered (React SPA). Playwright launches a headless Chromium browser, navigates to each job page, waits for the content to load, then extracts the job description text.

In [4]:
urls = [(j["title"], j.get("locationsText",""),
         f"{BASE_WD}/ServiceTitan{j['externalPath']}")
        for j in armenia]

async def scrape_details():
    records = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        for i, (title, location, url) in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] {title}")
            await page.goto(url, wait_until="networkidle", timeout=30000)
            try:
                await page.wait_for_selector('[data-automation-id="job-posting-details"]', timeout=10000)
            except:
                await page.wait_for_timeout(3000)
            content = await page.content()
            soup = BeautifulSoup(content, "html.parser")
            desc = ""
            for sel in ['[data-automation-id="job-posting-details"]',
                        '[data-automation-id="jobPostingDescription"]',
                        '.css-1t8ip2m']:
                el = soup.select_one(sel)
                if el:
                    desc = el.get_text("\n", strip=True)
                    break
            if not desc:
                body = soup.find("body")
                desc = body.get_text("\n", strip=True) if body else ""
            full_text = re.sub(r"\n{3,}", "\n\n", desc).strip()
            print(f"  full_text: {len(full_text)} chars")
            records.append({
                "source": "servicetitan", "source_url": url,
                "job_title": title, "company_name": "ServiceTitan",
                "location": location, "posting_date": "", "full_text": full_text,
            })
            await page.wait_for_timeout(1500)
        await browser.close()
    return records

records = asyncio.run(scrape_details())

[1/4] Senior Facilities Specialist
  full_text: 5636 chars
[2/4] Customer Success Manager
  full_text: 8158 chars
[3/4] Associate Data Quality Analyst
  full_text: 4959 chars
[4/4] Staff Site Reliability Engineer(Database)
  full_text: 5743 chars


## Step 3 — Save outputs

In [5]:
df = pd.DataFrame(records)
raw_path = RAW_DIR / "servicetitan_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

std_cols = ["source","source_url","job_title","company_name","location","posting_date","full_text"]
std_df = df[std_cols]
std_path = PROC_DIR / "servicetitan_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")
print()
print("=== FIELD COVERAGE ===")
for col in std_cols:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    print(f"  {col:20s}: {filled}/{len(std_df)} ({filled/len(std_df)*100:.0f}%)")
print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/servicetitan_jobs_raw.csv (4 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/servicetitan_jobs_standardized.csv (4 rows)

=== FIELD COVERAGE ===
  source              : 4/4 (100%)
  source_url          : 4/4 (100%)
  job_title           : 4/4 (100%)
  company_name        : 4/4 (100%)
  location            : 4/4 (100%)
  posting_date        : 0/4 (0%)
  full_text           : 4/4 (100%)

full_text — Min: 4959  Median: 5690  Max: 8158

Note: posting_date not exposed in Workday listing API response.
